# Finetuning GPT-2-XL

GPT-2 (XL):

  - 1.5 billion parameters
  - 48 layers
  - Hidden size: 1600
  - 25 attention heads

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
import torch
from transformers import pipeline

In [2]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

# Load dataset
dataset = load_dataset('vicclab/fairy_tales')

In [3]:
train_val = dataset["train"].train_test_split(
    test_size=0.2, seed=42)

In [4]:
dataset = DatasetDict({
    "train": train_val["train"],
    "validation": train_val["test"]
})

In [5]:
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Train size: 82878
Validation size: 20720


In [6]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2-xl')
tokenizer.pad_token = tokenizer.eos_token

In [7]:
def tokenize_function(examples):
    enc = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256   # ↑ important: longer context
    )
    return enc

In [8]:
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/82878 [00:00<?, ? examples/s]

Map:   0%|          | 0/20720 [00:00<?, ? examples/s]

In [9]:
tokenized_datasets = tokenized_datasets.filter(
    lambda x: len(x["input_ids"]) > 0
)

Filter:   0%|          | 0/82878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20720 [00:00<?, ? examples/s]

In [10]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [11]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained(
    "openai-community/gpt2-xl"
).to("cuda")

training_args = TrainingArguments(
    output_dir="models_gpt2_xl/results",
    eval_strategy="epoch",
    num_train_epochs=5,              # ↑ more signal at least 5
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="models_gpt2_xl/logs",
    save_strategy="epoch",
    report_to="none"
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.326000,3.523623
2,2.692800,3.463993
3,1.596900,3.914299
4,0.872400,4.383180
5,0.533900,4.718825


TrainOutput(global_step=83275, training_loss=1.794739139846623, metrics={'train_runtime': 9985.5507, 'train_samples_per_second': 33.358, 'train_steps_per_second': 8.34, 'total_flos': 5.8532459503872e+16, 'train_loss': 1.794739139846623, 'epoch': 5.0})

---